### Goal 

Practice extracting n-gram counts and learn how to use them to build a language model.

### Tasks 

- Split paragraphs in a dataset into word-like units called tokens, a process known as tokenization.
- Estimate the probabilities for an n-gram language model from a dataset.
- Use the n-gram language model to predict individual tokens and longer continuations.

In [1]:
import random 
from collections import defaultdict, Counter, deque
import textwrap # For formatting long text outputs
import pandas as pd 

### A) An example and calculations

In [2]:
africa_galore = pd.read_json("__assets/africa_galore.json") # Installed from "https://storage.googleapis.com/dm-educational/assets/ai_foundations/africa_galore.json"
africa_galore.head(3)

,category,name,description
0,Music,Afrobeat,"The Lagos air was thick with humidity, but the..."
1,Music,Highlife,The warm evening air in Accra was filled with ...
2,Music,Juju,The bustling streets of Lagos were alive with ...


In [3]:
dataset = africa_galore["description"]
print(f"The dataset consists of {dataset.shape[0]} paragraphs. \n")
print("Here are the first three paragraphs:\n")
for paragraph in dataset[:3]:
    # textwrap automatically adds linebreaks to make long texts more readable.
    formatted_paragraph = textwrap.fill(paragraph)
    print(f"{formatted_paragraph}\n")

The dataset consists of 232 paragraphs. 

Here are the first three paragraphs:

The Lagos air was thick with humidity, but the energy in the club was
electric. The band launched into a hypnotic Afrobeat groove, the drums
pounding out a complex polyrhythm, the horns blaring a soaring melody,
and the bass laying down a deep, funky foundation. A woman named Imani
moved effortlessly to the music, her body swaying in time with the
rhythm. The music seemed to flow through her, a powerful current of
energy and joy. All around her, people were dancing, singing, and
clapping, caught up in the infectious rhythm. The music was more than
just entertainment; it was a celebration of life, a connection to
their shared heritage, a vibrant expression of the soul of Lagos.

The warm evening air in Accra was filled with the lilting melodies of
Highlife music. At a small bar tucked away on a side street, a band
played, the guitars weaving intricate patterns, the horns adding a
bright, joyful counterpoint.

#### 1) Tokenization step 

In [4]:
def tokenize(text: str) -> list[str]:
    return text.split(" ") # Very simple whitespace tokenizer (can be improved)

print(tokenize(dataset[0]))

['The', 'Lagos', 'air', 'was', 'thick', 'with', 'humidity,', 'but', 'the', 'energy', 'in', 'the', 'club', 'was', 'electric.', 'The', 'band', 'launched', 'into', 'a', 'hypnotic', 'Afrobeat', 'groove,', 'the', 'drums', 'pounding', 'out', 'a', 'complex', 'polyrhythm,', 'the', 'horns', 'blaring', 'a', 'soaring', 'melody,', 'and', 'the', 'bass', 'laying', 'down', 'a', 'deep,', 'funky', 'foundation.', 'A', 'woman', 'named', 'Imani', 'moved', 'effortlessly', 'to', 'the', 'music,', 'her', 'body', 'swaying', 'in', 'time', 'with', 'the', 'rhythm.', 'The', 'music', 'seemed', 'to', 'flow', 'through', 'her,', 'a', 'powerful', 'current', 'of', 'energy', 'and', 'joy.', 'All', 'around', 'her,', 'people', 'were', 'dancing,', 'singing,', 'and', 'clapping,', 'caught', 'up', 'in', 'the', 'infectious', 'rhythm.', 'The', 'music', 'was', 'more', 'than', 'just', 'entertainment;', 'it', 'was', 'a', 'celebration', 'of', 'life,', 'a', 'connection', 'to', 'their', 'shared', 'heritage,', 'a', 'vibrant', 'expressio

#### 2) A little bit of leetcoding: Generating N-grams

In [5]:
def generate_ngrams(tokens: list[str], n: int) -> list[tuple[str, ...]]:
    ngrams = []
    currword = deque(maxlen=n) # Maxlen actually automatically does a pop front when appending new items
    for token in tokens:
        currword.append(token)
        if len(currword) == n:
            ngrams.append(tuple(currword))
    return ngrams

unigrams = generate_ngrams(tokenize(dataset[0]), 1)
bigrams = generate_ngrams(tokenize(dataset[0]), 2)
trigrams = generate_ngrams(tokenize(dataset[0]), 3)

print(f"First 5 unigrams: {unigrams[:5]}")
print(f"First 5 bigrams: {bigrams[:5]}")
print(f"First 5 trigrams: {trigrams[:5]}")

First 5 unigrams: [('The',), ('Lagos',), ('air',), ('was',), ('thick',)]
First 5 bigrams: [('The', 'Lagos'), ('Lagos', 'air'), ('air', 'was'), ('was', 'thick'), ('thick', 'with')]
First 5 trigrams: [('The', 'Lagos', 'air'), ('Lagos', 'air', 'was'), ('air', 'was', 'thick'), ('was', 'thick', 'with'), ('thick', 'with', 'humidity,')]


#### 3) Counting occurences of the N-grams

In [6]:
unigram_counts = Counter(unigrams) # Works because tuples are hashable
bigram_counts = Counter(bigrams)
trigram_counts = Counter(trigrams)
# Counter in python is a default dict of ints, and is built in O(n) for each n-gram list. 
# Equivalent to : defaultdict(int); for ngram in ngrams: d[ngram] += 1 
print(f"Most common unigrams: {unigram_counts.most_common(5)}")
print(f"Most common bigrams: {bigram_counts.most_common(5)}")
print(f"Most common trigrams: {trigram_counts.most_common(5)}")

Most common unigrams: [(('the',), 9), (('a',), 8), (('The',), 4), (('was',), 4), (('of',), 4)]
Most common bigrams: [(('in', 'the'), 2), (('rhythm.', 'The'), 2), (('The', 'music'), 2), (('The', 'Lagos'), 1), (('Lagos', 'air'), 1)]
Most common trigrams: [(('rhythm.', 'The', 'music'), 2), (('The', 'Lagos', 'air'), 1), (('Lagos', 'air', 'was'), 1), (('air', 'was', 'thick'), 1), (('was', 'thick', 'with'), 1)]


#### 4) Converting the entire dataset into N-grams, and getting their counts (from which we can estimate probabilities )

In [7]:
def get_ngram_counts(dataset: list[str], n:int ) -> dict[str, Counter] : 
    ngram_counts = defaultdict(Counter)
    for text in dataset:
        tokens = tokenize(text)
        ngrams = generate_ngrams(tokens, n)
        for ngram in ngrams:
            # The before sentence context is all tokens except the last one
            context = ngram[:-1]  
            # How many times we saw each next token after this context
            next_token = ngram[-1] 
            ngram_counts[" ".join(context)][next_token] += 1 # Increment count of seeing next_token after context
    return dict(ngram_counts)

example_data = [
    "This is an example sentence.",
    "Another example sentence.",
    "Split a sentence."
]

get_ngram_counts(example_data, 3)

{'This is': Counter({'an': 1}),
 'is an': Counter({'example': 1}),
 'an example': Counter({'sentence.': 1}),
 'Another example': Counter({'sentence.': 1}),
 'Split a': Counter({'sentence.': 1})}

In [8]:
# Using bigram counts on the africa_galore dataset
bigram_counts = get_ngram_counts(dataset, 2) 
# Let's display it as a table 
bigram_counts_matrix = {
    context: dict(counts) for context, counts in bigram_counts.items()
}
bigram_data_frame = pd.DataFrame.from_dict(
    bigram_counts_matrix, orient="index").fillna(0)
display(bigram_data_frame.head(10))
# Counting zeros to see sparsity
zero_count = (bigram_data_frame == 0).sum().sum()
print(
    f"Number of bigrams with a count of 0: {zero_count:,}"
    f" ({zero_count/bigram_data_frame.size * 100:.2f}%)"
    " Very Sparse!"
)

,Lagos,band,music,warm,Highlife,bustling,Dakar,Mbalax,Kinshasa,Soukous,...,"kudu,","mph),",Ostriches,Antarctic,plumage,surface.,(Spheniscus,demersus).,breed,Bay
The,1.0,1.0,4.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
of,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
the,0.0,1.0,0.0,4.0,0.0,4.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
a,0.0,1.0,0.0,6.0,0.0,7.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
with,0.0,0.0,1.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Afrobeat,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
and,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
for,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Mbalax,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Senegalese,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Number of bigrams with a count of 0: 26,606,550 (99.95%) Very Sparse!


In [9]:
# Trying with trigrams now
trigram_counts = get_ngram_counts(dataset, 3)
trigram_counts_matrix = {
    context: dict(counts) for context, counts in trigram_counts.items()
}
trigram_data_frame = pd.DataFrame.from_dict(
    trigram_counts_matrix, orient="index").fillna(0)
display(trigram_data_frame.head(10))
print(trigram_data_frame.shape, "Huge DataFrame!")
# Counting zeros to see sparsity
zero_count = (trigram_data_frame == 0).sum().sum()
print(
    f"Number of trigrams with a count of 0: {zero_count:,}"
    f" ({zero_count/trigram_data_frame.size * 100:.2f}%)"
    " Even More Sparse!"
)

,air,was,thick,thin,always,"quiet,",filled,alive,with,"humidity,",...,plumage,water's,surface.,penguin,(Spheniscus,demersus).,penguins,breed,Algoa,Bay
The Lagos,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
in the,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
and the,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
warm evening,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
vegetables. The,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
filled the,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
river. The,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
where the,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
texture. The,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Ali. The,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


(13411, 5142) Huge DataFrame!
Number of trigrams with a count of 0: 68,942,324 (99.98%) Even More Sparse!


We should consider a better data structure to avoid exploding (with big LLMs we have N > 3 grams, and a lot more tokens), and handles sparsity well. 

We can estimate the conditional probabilities from such huge tables.
So we have $P(token|context)$ is the probability we need to compute. By simple conditional probabilities we have : $$P(token | context) = \frac{P(token \cap context)}{P(context)}$$. Actually, we can consider $token \cap context$ as an N-grams (concatenation of token and context). It is obvious that in this case, context is an (N-1)-gram. 

In [10]:
numerator = get_ngram_counts(dataset, 3) # Numerator is token + context 
denominator = get_ngram_counts(dataset, 2) # Denominator is context only 

context = "a staple" # Or "a"
numerator[context] # In case context is one element, use denominator[context]

Counter({'in': 6,
         'dish': 2,
         'food': 1,
         'throughout': 1,
         'of': 1,
         'at': 1,
         'beverage': 1})

In [11]:
context= "a staple"
# Computing the cardinal of all possible values we have seen in the data (with occurrences)
bigram_count_a_staple = sum(numerator[context].values())
bigram_counts["a"]["staple"], bigram_count_a_staple # Same value because a staple appears the same amount of time (13)

(13, 13)

#### 5) Computing the probabilities and building the N-gram model

In [12]:
def build_ngram_model(dataset:list[str], n:int) -> dict[str, dict[str, float]]:
    ngram_model = {}
    # We count the n-grams in the dataset 
    ngram_counts = get_ngram_counts(dataset, n)
    # Now we have contexts linked to counter of next tokens
    for context, next_token_counts in ngram_counts.items():
        total_count = sum(next_token_counts.values())
        # Now we can compute probabilities for each next token
        ngram_model[context] = {
            token: count / total_count 
            for token, count in next_token_counts.items()
        }
    return ngram_model

test_dataset = ["Table Mountain is tall.", "Table Mountain is beautiful."]
ngram_model = build_ngram_model(test_dataset, 3)
ngram_model

{'Table Mountain': {'is': 1.0},
 'Mountain is': {'tall.': 0.5, 'beautiful.': 0.5}}

In [13]:
trigram_model = build_ngram_model(dataset, 3)
print(f"P(token | \"as it\") = {trigram_model['as it']}")
print(f"P(token | \"as they\") = {trigram_model['as they']}")
print(f"P(token | \"The name\") = {trigram_model['The name']}")
print(f"P(token | \"a staple\") = {trigram_model['a staple']}")
print(f"P(token | \"nonexistent context\") = {trigram_model.get('nonexistent context', 'Context not found')}")

P(token | "as it") = {'is': 0.6666666666666666, 'receives': 0.3333333333333333}
P(token | "as they") = {'were': 1.0}
P(token | "The name") = {'means': 0.6666666666666666, "'Etosha'": 0.3333333333333333}
P(token | "a staple") = {'food': 0.07692307692307693, 'in': 0.46153846153846156, 'dish': 0.15384615384615385, 'throughout': 0.07692307692307693, 'of': 0.07692307692307693, 'at': 0.07692307692307693, 'beverage': 0.07692307692307693}
P(token | "nonexistent context") = Context not found


#### 6) Let's try running this model

In [14]:
# Simple code : 
def predict_next_token(context: str, ngram_model: dict[str, dict[str, float]]) -> str:
    if context not in ngram_model:
        return "Context not found in model."
    next_token_probs = ngram_model[context]
    # Choose the next token based on the probabilities
    tokens = list(next_token_probs.keys())
    probabilities = list(next_token_probs.values())
    predicted_token = random.choices(tokens, weights=probabilities, k=1)[0]
    return predicted_token

context = "a staple"
predicted_token = predict_next_token(context, trigram_model)
print(f"Predicted next token after '{context}': {predicted_token}")

Predicted next token after 'a staple': at


In [15]:
# n words generations 
def generate_text(start_context: str, ngram_model: dict[str, dict[str, float]], num_tokens: int, n: int) -> str:
    context = start_context.split(" ")[- (n-1) :]  # Start with the last n-1 words of the context
    context = " ".join(context)
    generated_tokens = []
    for _ in range(num_tokens):
        next_token = predict_next_token(context, ngram_model)
        if next_token == "Context not found in model.":
            generated_tokens.append("[END PREMATURELY]")  # Unknown token
            break
        generated_tokens.append(next_token)
        # Update context by removing the first word and adding the new token
        context_words = context.split(" ")
        context_words.append(next_token)
        context = " ".join(context_words[1:])  # Keep the last n-1 words as context
    return " ".join(generated_tokens)

start_context = "a staple"
generated_text = generate_text(start_context, trigram_model, 100, 3)
print(f"Generated text after '{start_context}':")
start_context+" "+generated_text

Generated text after 'a staple':


"a staple throughout Southern Africa. It is a large mug filled with the sounds of birdsong and the specific forms of the largest language family also has a mildly sweet, non-alcoholic drink from northern Nigeria made from purÃ©ed black-eyed peas. The batter is mixed with milk or water, sometimes spiced with ginger and cardamom, that warmed them from the Tswana word Kgalagadi meaning 'the great thirst,' aptly describes this expansive landscape characterized by hot, dry climate with warm temperatures and higher rainfall. Southern Africa is also a key stronghold for the kings of the iconic Ring-tailed Lemur, are matriarchal, with females dominating"

In [16]:
# Let us try on a different context not ever seen before
prompt = "Jide was hungry so she went looking for"
generated_text = generate_text(prompt, trigram_model, 50, 3)
print(f"Generated text after '{prompt}': \n {prompt + ' ' + generated_text}")

Generated text after 'Jide was hungry so she went looking for': 
 Jide was hungry so she went looking for the millions of years. Unusually for a touch of variety and excitement to plain water. [END PREMATURELY]


Okey so it does generate text, now does it really make sense ? Not quite in some cases. N-grams only see n previous words, and so diverging from the main theme of the sentence is very likely. 

In [17]:
# Should be even worse with bigrams

bigram_model = build_ngram_model(dataset, 2)
prompt = "Jide was hungry so she went looking for"
generated_text = generate_text(prompt, bigram_model, 50, 2)
print(f"Generated text after '{prompt}': \n {prompt + ' ' + generated_text}")

Generated text after 'Jide was hungry so she went looking for': 
 Jide was hungry so she went looking for its blend of brave ancestors, spirits, and Central African wild dog. It was a meal; it can soar above and smoked fish. It is rich and bushveld habitats. Kudus are small classroom. The air was teaching Esi loved exploring the incredible biodiversity. Its distinctive black spots and form. They were


In [18]:
# How about more grams 
tengram_model = build_ngram_model(dataset, 10)
prompt = "Jide was hungry so she went looking for"
generated_text = generate_text(prompt, tengram_model, 50, 10)
print(f"Generated text after '{prompt}': \n {prompt + ' ' + generated_text}") # End of result fast because it did  not find context

Generated text after 'Jide was hungry so she went looking for': 
 Jide was hungry so she went looking for [END PREMATURELY]


The more context N-gram we have, the higher the chance to not actually find the context already visited, but will be more strict in text generations. Actually, let's try that. 

In [19]:
prompt = "The Lagos air was thick with humidity, but the energy in" # First paragraph
generated_text = generate_text(prompt, tengram_model, 50, 10)
print(f"Generated text after '{prompt}': \n {prompt + ' ' + generated_text}") 
print(dataset[0].startswith(prompt +' '+ generated_text)) # The exact match

Generated text after 'The Lagos air was thick with humidity, but the energy in': 
 The Lagos air was thick with humidity, but the energy in the club was electric. The band launched into a hypnotic Afrobeat groove, the drums pounding out a complex polyrhythm, the horns blaring a soaring melody, and the bass laying down a deep, funky foundation. A woman named Imani moved effortlessly to the music, her body swaying in time with the
True


This is exactly the problem of data sparsity because we see less and less links between context and new tokens. N-grams are very sensitive to small datasets, and so we needed a better way to generate text. 
We will work with LLMs that are far better in this task. 